# 13 — MCTS Evaluation
Runs MCTSPromptOptimizer on GSM8K 500-subset and 120 WSU domain cases.
Results saved to `data/results/mcts_results.json` and appended to `data/results/comparison_metrics.csv`.

In [ ]:
import sys, json, time, csv
from pathlib import Path

REPO_ROOT = Path("../..")
sys.path.insert(0, str(REPO_ROOT / "src"))

from evaluation.benchmarks import BenchmarkLoader
from evaluation.metrics import EvaluationMetrics
from llm.cache import ResponseCache
from prompts.template import PromptTemplate
from prompts.mutations import PromptMutator
from search.mcts import MCTSPromptOptimizer

cache = ResponseCache(db_path=str(REPO_ROOT / "data" / "cache" / "responses.db"))
loader = BenchmarkLoader(cache_dir=REPO_ROOT / "data" / "benchmarks")
results_dir = REPO_ROOT / "data" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
gsm8k = loader.load_gsm8k(subset_size=500)
wsu_path = REPO_ROOT / "data" / "benchmarks" / "test_cases.json"
wsu = json.loads(wsu_path.read_text())[:120] if wsu_path.exists() else []

baseline_path = results_dir / "baseline_results.json"
baseline_map = {}
if baseline_path.exists():
    for row in json.loads(baseline_path.read_text()):
        if row["method"] == "zero_shot":
            baseline_map[row["dataset"]] = row["accuracy"]

In [ ]:
from llm.base import BaseLLMClient

class CachedMockLLM(BaseLLMClient):
    def complete(self, prompt, **kwargs):
        cached = cache.get(prompt, model="claude-3-haiku", temperature=0.0)
        if cached:
            return cached, {"input_tokens": 0, "output_tokens": 0}
        response = "[mock]"
        cache.set(prompt, model="claude-3-haiku", temperature=0.0, response=response)
        return response, {"input_tokens": len(prompt.split()), "output_tokens": 5}

llm = CachedMockLLM()

In [ ]:
def run_mcts(dataset, ds_name):
    template = PromptTemplate(task="Answer the following question.")
    mutator = PromptMutator()
    optimizer = MCTSPromptOptimizer(
        iterations=30,
        llm_client=llm,
        mutator=mutator,
    )
    val_set = [{"input": ex["question"], "expected": ex["answer"]} for ex in dataset[:50]]
    t0 = time.time()
    best_template = optimizer.search(template, val_set)
    runtime = time.time() - t0

    predictions, usage = [], {"input_tokens": 0, "output_tokens": 0}
    for ex in dataset:
        prompt = best_template.render(question=ex["question"])
        resp, u = llm.complete(prompt)
        predictions.append(resp)
        usage["input_tokens"]  += u["input_tokens"]
        usage["output_tokens"] += u["output_tokens"]

    ground_truth = [ex["answer"] for ex in dataset]
    acc = EvaluationMetrics.accuracy(predictions, ground_truth)
    baseline = baseline_map.get(ds_name, 0.0)
    improvement = EvaluationMetrics.normalized_score(acc, baseline)

    return {
        "algorithm": "mcts", "dataset": ds_name,
        "accuracy": round(acc, 4),
        "improvement_over_baseline": round(improvement, 4),
        "api_calls_used": len(dataset) + 30,
        "runtime_seconds": round(runtime, 2),
    }

rows = []
for ds_name, ds in [("gsm8k", gsm8k)] + ([("wsu", wsu)] if wsu else []):
    row = run_mcts(ds, ds_name)
    rows.append(row)
    print(row)

(results_dir / "mcts_results.json").write_text(json.dumps(rows, indent=2))
csv_path = results_dir / "comparison_metrics.csv"
fieldnames = ["algorithm","dataset","accuracy","improvement_over_baseline","api_calls_used","runtime_seconds"]
write_header = not csv_path.exists()
with open(csv_path, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    if write_header:
        writer.writeheader()
    writer.writerows(rows)
print("Done.")